In [1]:
import pandas as pd
import numpy as np
import mssql_python
from pathlib import Path


In [2]:
# Data path
base_path = Path('C:/Users/User/OneDrive/Desktop/UN')
data_path = Path('C:/Users/User/OneDrive/Desktop/UN/Data')
data_path = Path('C:/Users/alfon/Desktop/UN/Data')


data_files_dict = {'index_1030' : ['index_1030 Gender Development Index.xlsx', '2025_metadata_Gender Development Index 2025.xlsx'],
                   'index_9' : ['index_9 EGDI.xlsx', '20251102_metadata_E-Government Development Index_2024.xlsx', 'raw_EGDI_2024.xlsx'],
                   'index_1032': ['20260710_historical data_index_1032_SDG 2026.xlsx', '20260710_metadata_Sustainable_Development_Index_2026.xlsx', 'raw_SDG_2026.xlsx'],

                    # Somalia indices
                   'index_12' : ['index_12 E-participation Index.xlsx', '', 'raw_EPI_2024.xlsx'],
                   'index_1049' : ['index_1049.xlsx', '', 'raw_IIAG_2024.xlsx'],
                   'index_1037' : ['index_1037.xlsx', '', 'raw_GOVTECH_2022.xlsx'],
                   'index_1022': ['index_1022 Women Peace and security Index.xlsx','','raw_WPS_2025.xlsx'],
                   
                   # Somalia second batch
                   'index_1050': ['index_1050 Global Hunger.xlsx','','raw_GHI.xlsx'],
                   'index_5': ['index_5 ICT Developement Index.xlsx','','raw_ITU_2024.xlsx'],
                   'index_1036': ['20262205_historical data_index_1036_WPF_2026.xlsx','','raw_WPF_2026.xlsx'],

                   # Third
                   'index_1027': ['index_1027 Human Development Index.xlsx','','raw_HDI.xlsx'],
                   'index_1030': ['index_1030 Gender Development Index.xlsx','','raw_GDI_2024.xlsx'],
                   'index_1031': ['index_1031 Planetary adjusted HDI.xlsx','','raw_PPHDI.xlsx'],
                   'index_1035': ['20260303_Historical data _WBL2.0_2026.xlsx','','raw_WBL2.0_2026.xlsx'],
                   # None,
                   'index_1026': ['index_1026 Open data Policies.xlsx','','raw_ODP_2022.xlsx'],
                   'index_1053': ['20260721_historical data_index_1053_BTI2026.xlsx','','raw_BTI_2026.xlsx'],
                   'index_1021': ['index_1021.xlsx','','raw_WBL 1.0.xlsx'],
                   'index_1052': ['20260717_historical data_index_1052_WPJ2026.xlsx','','raw_WJP_2025.xlsx'],
                   'index_1034': ['index_1034.xlsx','','raw_EnviPI_2026.xlsx'],
            
                  }
#Human Development Index (index_1027)
#Gender Development Index (index_1030)
#Planetary adjusted Human development Index (index_1031)
#Women Business and the Law 2.0 (index_1035)

index_name = 'index_9'
index_data_files = data_files_dict[index_name]

index_historical_kpi_data_path = Path(data_path) / 'Historical' / index_data_files[0]
#index_metadata_path = Path(data_path) / 'Metadata' / index_data_files[1]
index_simulator_data_path = Path(data_path) / 'Simulator Data' / index_data_files[2]
diagnosis_assessment_template_path = Path(data_path) / 'Reference templates' / 'Data_Diagnosis_Assessment_template.xlsx'

# Reference template
#diagnosis_assessment_template = pd.read_excel(diagnosis_assessment_template_path)

# Read raw data
historical_kpi_data_raw = pd.read_excel(index_historical_kpi_data_path)
#index_metadata_raw = pd.read_excel(index_metadata_path)
#index_metadata_raw['Arab Countries'] = index_metadata_raw['Arab Countries'].str.strip()
simulator_data_raw = pd.read_excel(index_simulator_data_path)

country_iso = pd.read_excel(Path(data_path) / 'metadata_SDG_2026.xlsx', sheet_name='country')
country_iso['country'] = country_iso['country'].str.strip()
country_iso['ISO'] = country_iso['ISO'].str.strip()
# - data correction
if index_name == 'index_1032':
    #index_metadata_raw['Index_ID'] = '1032'
    #index_metadata_raw = index_metadata_raw.merge(country_iso, how='left', left_on='Arab Countries', right_on='country')
    # make first row columns names
    historical_kpi_data_raw.columns = historical_kpi_data_raw.iloc[0]
    #index_data_raw = index_data_raw[1:]


# Reference Data
escwa_members = ['DZA','BHR','COM','DJI','EGY','IRQ','JOR','KWT','LBN','LBY','MRT',
                 'MAR','OMN','PSE','QAT','SAU','SOM','SDN','SYR','TUN','ARE','YEM']


kpi_key = (simulator_data_raw.iloc[0]
           .rename('Series Name')
           .rename_axis('KPI ID')
           .reset_index()
           .dropna(subset=['Series Name']))
kpi_key['Series Name'] = kpi_key['Series Name'].astype(str).str.strip()
kpi_key['KPI ID'] = kpi_key['KPI ID'].astype(str).str.strip()
######## Sim data ###################
simulator_data = simulator_data_raw.copy()
simulator_data = simulator_data[1:].reset_index(drop=True)

pivot_names = ['Country', 'ISO', 'Region', 'Sub-region', 'Income Type']
# Convert to long format
simulator_data = simulator_data.melt(id_vars=pivot_names, 
                         var_name='KPI ID', value_name='data_report')      # convert wide to long
simulator_data['KPI ID'] = simulator_data['KPI ID'].astype(str).str.strip()
simulator_data = simulator_data.merge(kpi_key,how='left')
simulator_data = simulator_data[['Country', 'ISO', 'Series Name', 'KPI ID', 'data_report']]
simulator_data['data_report'] = pd.to_numeric(
    simulator_data['data_report'], errors='coerce'
)
########## New ############


# Index data minor reformatting and filtering
historical_kpi_data = historical_kpi_data_raw.copy()
historical_kpi_data = historical_kpi_data.rename(columns={'Unnamed: 0':'KPI ID','Unnamed: 1':'Series Name','Unnamed: 2':'year_historical_automated',
                                        'year':'year_historical_automated', 'Series name':'Series Name', 'KeyID':'KPI ID'})
historical_kpi_data = historical_kpi_data.iloc[1:]
# - Values to change, match simulator values
replacement_dict = {
    'Mean years of schooling': 'Mean Year of Schooling',
    'Expected years of schooling': 'Expected Year of Schooling'
}
historical_kpi_data['Series Name'] = historical_kpi_data['Series Name'].replace(replacement_dict)
# - long format
historical_kpi_data_long = historical_kpi_data.melt(id_vars=['KPI ID','Series Name','year_historical_automated'], 
                                           var_name='ISO', value_name='data_historical_automated')        # convert wide to long
#historical_kpi_data_long = historical_kpi_data_long[historical_kpi_data_long['ISO'].isin(escwa_members)]
# - format values for consistency
historical_kpi_data_long['data_historical_automated'] = pd.to_numeric(
    historical_kpi_data_long['data_historical_automated'], errors='coerce').astype('Float64')
historical_kpi_data_long['KPI ID'] = historical_kpi_data_long['KPI ID'].astype(str).str.strip()
historical_kpi_data_long['year_historical_automated'] = historical_kpi_data_long['year_historical_automated'].astype('Int64')

series_to_kpi = historical_kpi_data_long[['Series Name', 'KPI ID']].copy().drop_duplicates().reset_index(drop=True)

In [3]:
kpi_list = simulator_data['KPI ID'].unique().tolist()

In [4]:
import os
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("No .env file found in the current directory or its parents")
load_dotenv(env_path, override=True)

def get_connection_string(env: str = "dev") -> str:
    prefix = env.upper()
    try:
        server = os.environ[f"{prefix}_DB_SERVER"]
        database = os.environ[f"{prefix}_DB_NAME"]
        user = os.environ[f"{prefix}_DB_USER"]
        password = os.environ[f"{prefix}_DB_PASSWORD"]
    except KeyError as e:
        raise RuntimeError(f"Missing environment variable: {e.args[0]}") from None

    return (
        f"Server={server};"
        f"Database={database};"
        f"UID={user};"
        f"PWD={password};"
        "TrustServerCertificate=Yes"
    )
    
connection_string = get_connection_string(env='prod')

In [5]:
# Pull current simulator report data
placeholders = ",".join("?" for _ in kpi_list)
sim_data_query = f"""
    WITH ranked AS (
        SELECT
            c.Name AS Country,
            c.ISO,
            k.Name AS [Series Name],
            d.KPIID AS [KPI ID],
            d.Value AS data_report,
            d.[year] AS data_report_year,
            RANK() OVER (
                PARTITION BY c.ISO, d.KPIID
                ORDER BY d.[year] DESC
            ) AS rn
        FROM KPIData AS d
        LEFT JOIN Countries AS c
            ON d.CountryID = c.keyID
        LEFT JOIN KPIs AS k
            ON d.KPIID = k.KeyID
        WHERE d.KPIID IN ({placeholders})
    )
    SELECT Country, ISO, [Series Name], [KPI ID], data_report, data_report_year
    FROM ranked
    WHERE rn = 1;
"""

with mssql_python.connect(connection_string) as conn:
    with conn.cursor() as cursor:
        cursor.execute(sim_data_query, kpi_list)
        columns = [col[0] for col in cursor.description]
        rows = cursor.fetchall()
sim_data = pd.DataFrame.from_records(rows, columns=columns)


In [7]:
placeholders = ",".join("?" for _ in kpi_list)
hist_data_query = f"""
    SELECT 
        h.KPIID AS 'KPI ID',
        k.Name  AS 'Series Name',
        h.Year AS year_historical_automated,
        c.ISO,
        h.Value AS data_historical_automated
    FROM HistoricalKPI AS h
    LEFT JOIN Countries AS c
        ON h.CountryID = c.keyID
    LEFT JOIN KPIs AS k
        ON h.KPIID = k.KeyID
    WHERE h.KPIID IN ({placeholders});
"""
with mssql_python.connect(connection_string) as conn:
    with conn.cursor() as cursor:
        cursor.execute(hist_data_query, kpi_list)
        columns = [col[0] for col in cursor.description]
        rows = cursor.fetchall()
hist_data = pd.DataFrame.from_records(rows, columns=columns)

min_max_year = (
    hist_data.groupby(['KPI ID'])['year_historical_automated']
      .agg(['min', 'max'])
      .reset_index()
)

# Make sure the year columns are integers on both sides
hist_data['year_historical_automated'] = hist_data['year_historical_automated'].astype(int)
min_max_year[['min', 'max']] = min_max_year[['min', 'max']].astype(int)

# Every KPI  / country combination that appears in the data
kpi_iso = hist_data[['KPI ID','Series Name', 'ISO']].drop_duplicates()

# Attach each KPI's year range and expand it into one row per year
grid = kpi_iso.merge(min_max_year, on=['KPI ID'], how='left')
grid['year_historical_automated'] = [np.arange(lo, hi + 1) for lo, hi in zip(grid['min'], grid['max'])]
grid = grid.explode('year_historical_automated').drop(columns=['min', 'max'])
grid['year_historical_automated'] = grid['year_historical_automated'].astype(int)

# Bring in the real data; missing years get NaN
filled = (
    grid.merge(hist_data,
               on=['KPI ID', 'ISO','Series Name', 'year_historical_automated'],
               how='left')
        .sort_values(['KPI ID', 'ISO', 'year_historical_automated'])
        .reset_index(drop=True)
)
# NA do not show up here, if there is no data for KPI 271 / ISO 'ATG' then it will not exist
# - identify max and min year value for KPI 
# - based on these two values fill in year values with records that are np.nan

NameError: name 'rename_map' is not defined

In [5]:
def get_latest_hist_data(connection_string: str, kpi_list: list) -> pd.DataFrame:
    """Return historical data per ISO and KPI ID, with every year between each
    KPI's min and max year present (missing years filled with NaN)."""
    empty_cols = ["KPI ID", "Series Name", "year_historical_automated",
                  "ISO", "data_historical_automated"]
    if not kpi_list:
        return pd.DataFrame(columns=empty_cols)

    placeholders = ",".join("?" for _ in kpi_list)
    query = f"""
        SELECT
            h.KPIID AS [KPI ID],
            k.Name  AS [Series Name],
            h.[Year] AS year_historical_automated,
            c.ISO,
            h.Value AS data_historical_automated
        FROM HistoricalKPI AS h
        LEFT JOIN Countries AS c ON h.CountryID = c.keyID
        LEFT JOIN KPIs AS k ON h.KPIID = k.KeyID
        WHERE h.KPIID IN ({placeholders});
    """

    with mssql_python.connect(connection_string) as conn:
        with conn.cursor() as cursor:
            cursor.execute(query, list(kpi_list))
            columns = [col[0] for col in cursor.description]
            rows = cursor.fetchall()

    hist_data = pd.DataFrame.from_records(rows, columns=columns)
    if hist_data.empty:
        return pd.DataFrame(columns=empty_cols)

    # Drop rows without a year so the int conversion doesn't fail
    hist_data = hist_data.dropna(subset=["year_historical_automated"])
    hist_data["year_historical_automated"] = hist_data["year_historical_automated"].astype(int)

    # Min and max year per KPI
    min_max_year = (
        hist_data.groupby("KPI ID")["year_historical_automated"]
                 .agg(["min", "max"])
                 .reset_index()
    )

    # Every KPI / Series / country combination that appears in the data
    kpi_iso = hist_data[["KPI ID", "Series Name", "ISO"]].drop_duplicates()

    # Attach each KPI's year range and expand it into one row per year
    grid = kpi_iso.merge(min_max_year, on="KPI ID", how="left")
    grid["year_historical_automated"] = [
        np.arange(lo, hi + 1) for lo, hi in zip(grid["min"], grid["max"])
    ]
    grid = grid.explode("year_historical_automated").drop(columns=["min", "max"])
    grid["year_historical_automated"] = grid["year_historical_automated"].astype(int)

    # Bring in the real data; missing years get NaN
    return (
        grid.merge(hist_data,
                   on=["KPI ID", "Series Name", "ISO", "year_historical_automated"],
                   how="left")
            .sort_values(["KPI ID", "ISO", "year_historical_automated"])
            .reset_index(drop=True)
    )

    

In [6]:
hist_data = get_latest_hist_data(connection_string, kpi_list)


In [7]:
hist_data.to_clipboard()

In [22]:
min_max_year = (
    hist_data.groupby(['KPI ID'])['year_historical_automated']
      .agg(['min', 'max'])
      .reset_index()
)

In [34]:
min_max_year = (
    hist_data.groupby(['KPI ID'])['year_historical_automated']
      .agg(['min', 'max'])
      .reset_index()
)

# Make sure the year columns are integers on both sides
hist_data['year_historical_automated'] = hist_data['year_historical_automated'].astype(int)
min_max_year[['min', 'max']] = min_max_year[['min', 'max']].astype(int)

# Every KPI  / country combination that appears in the data
kpi_iso = hist_data[['KPI ID','Series Name', 'ISO']].drop_duplicates()

# Attach each KPI's year range and expand it into one row per year
grid = kpi_iso.merge(min_max_year, on=['KPI ID'], how='left')
grid['year_historical_automated'] = [np.arange(lo, hi + 1) for lo, hi in zip(grid['min'], grid['max'])]
grid = grid.explode('year_historical_automated').drop(columns=['min', 'max'])
grid['year_historical_automated'] = grid['year_historical_automated'].astype(int)

# Bring in the real data; missing years get NaN
filled = (
    grid.merge(hist_data,
               on=['KPI ID', 'ISO','Series Name', 'year_historical_automated'],
               how='left')
        .sort_values(['KPI ID', 'ISO', 'year_historical_automated'])
        .reset_index(drop=True)
)

In [35]:
filled

,KPI ID,Series Name,ISO,year_historical_automated,data_historical_automated
0,271,Mobile cellular telephone subscriptions per 10...,AFG,1988,NaN
1,271,Mobile cellular telephone subscriptions per 10...,AFG,1989,NaN
2,271,Mobile cellular telephone subscriptions per 10...,AFG,1990,0.00
3,271,Mobile cellular telephone subscriptions per 10...,AFG,1991,0.00
4,271,Mobile cellular telephone subscriptions per 10...,AFG,1992,0.00
...,...,...,...,...,...
60041,1942,Percentage of Fixed-broadband Internet basket ...,ZWE,2020,NaN
60042,1942,Percentage of Fixed-broadband Internet basket ...,ZWE,2021,18.30
60043,1942,Percentage of Fixed-broadband Internet basket ...,ZWE,2022,9.45
60044,1942,Percentage of Fixed-broadband Internet basket ...,ZWE,2023,12.80


In [15]:
min_max_year.to_clipboard()

In [8]:
hist_data[(hist_data['ISO']=='ATG')&(hist_data['KPI ID']==271)].to_clipboard()

In [10]:
placeholders = ",".join("?" for _ in kpi_list)
hist_data_query = f"""
    SELECT TOP 10 *
    FROM HistoricalKPI
    WHERE KPIID IN ({placeholders});
"""
with mssql_python.connect(connection_string) as conn:
    with conn.cursor() as cursor:
        cursor.execute(hist_data_query, kpi_list)
        columns = [col[0] for col in cursor.description]
        rows = cursor.fetchall()
hist_data = pd.DataFrame.from_records(rows, columns=columns)
hist_data

,KPIID,CountryID,Year,Category,Value
0,271,1,1990,1,0.0
1,271,1,1991,1,0.0
2,271,1,1992,1,0.0
3,271,1,1993,1,0.0
4,271,1,1994,1,0.0
5,271,1,1995,1,0.0
6,271,1,1996,1,0.0
7,271,1,1997,1,0.0
8,271,1,1998,1,0.0
9,271,1,1999,1,0.0


In [16]:
hist_data.duplicated(['KPI ID', 'ISO', 'year_historical_automated']).sum()

np.int64(0)

In [53]:
result = sim_data.groupby(['ISO','KPI ID'])['data_report_year'].unique()
result

ISO  KPI ID
AFG  271       [2024]
     272       [2024]
     274       [2024]
     279       [2024]
     280       [2024]
                ...  
ZWE  1240      [2024]
     1241      [2024]
     1940      [2024]
     1941      [2024]
     1942      [2024]
Name: data_report_year, Length: 2895, dtype: object

In [49]:
sim_data[sim_data['KPI ID']==272].to_clipboard()

In [36]:
    LEFT JOIN KPIs AS k
        ON k.KeyID = d.KPIID

,Name,ISO,CountryID,KPIID,Value,year
0,Afghanistan,AFG,1,271,59.12,2020
1,Albania,ALB,2,271,94.18,2020
2,Algeria,DZA,3,271,111.66,2020
3,Andorra,AND,4,271,107.28,2020
4,Angola,AGO,5,271,43.13,2020
...,...,...,...,...,...,...
6557,Venezuela,VEN,188,282,11.1,2024
6558,Viet Nam,VNM,189,282,8.4,2024
6559,Yemen,YEM,190,282,3.2,2024
6560,Zambia,ZMB,193,282,7.2,2024


In [28]:
    SELECT h.*,
           c.Name  AS CountryName,
           c.ISO   AS CountryISO,
           k.KeyID AS KPIKeyID,
           k.Name  AS KPIName
    FROM HistoricalKPI AS h
    LEFT JOIN Countries AS c
        ON h.CountryID = c.keyID

,CountryID,KPIID,Value,year
0,1,271,59.12,2020
1,2,271,94.18,2020
2,3,271,111.66,2020
3,4,271,107.28,2020
4,5,271,43.13,2020
...,...,...,...,...
6557,188,282,11.1,2024
6558,189,282,8.4,2024
6559,190,282,3.2,2024
6560,193,282,7.2,2024


In [22]:
hist_data_query = f"""
    SELECT h.*,
           c.Name  AS CountryName,
           c.ISO   AS CountryISO,
           k.KeyID AS KPIKeyID,
           k.Name  AS KPIName
    FROM HistoricalKPI AS h
    LEFT JOIN Countries AS c
        ON h.CountryID = c.keyID
    LEFT JOIN KPIs AS k
        ON h.KPIID = k.KeyID
    WHERE h.KPIID IN ({placeholders});

,KeyID,CountryID,KPIID,Category,Value,year,Rank
0,75380,1,271,1,59.12,2020,None
1,75381,2,271,1,94.18,2020,None
2,75382,3,271,1,111.66,2020,None
3,75383,4,271,1,107.28,2020,None
4,75384,5,271,1,43.13,2020,None
5,75385,6,271,1,120,2020,None
6,75386,7,271,1,120,2020,None
7,75387,8,271,1,120,2020,None
8,75388,9,271,1,113.58,2020,None
9,75389,10,271,1,120,2020,None


In [2]:
# Prod connection test
server_name='sqldb-p-server-001.database.windows.net'
database_name='ispar'
username='ispar_user_read'
pwd='TAescwa2025##'

connection_string = (
    f"Server={server_name};"
    f"Database={database_name};"
    f"UID={username};"
    f"PWD={pwd};"
    "TrustServerCertificate=Yes"
)

# Establish connection
with mssql_python.connect(connection_string) as conn:
    # Create a cursor object
    with conn.cursor() as cursor:
        # Execute statement to look up user tables
        cursor.execute(
            """
            SELECT TABLE_SCHEMA, TABLE_NAME 
            FROM INFORMATION_SCHEMA.TABLES 
            WHERE TABLE_TYPE = 'BASE TABLE';
            """
        )
        # Fetch and print table names
        for row in cursor.fetchall():
            print(f"Schema: {row[0]}, Table: {row[1]}")
            

ProgrammingError: Driver Error: Syntax error or access violation; DDBC Error: [Microsoft][SQL Server]Cannot open server 'sqldb-p-server-001' requested by the login. Client with IP address '172.58.240.245' is not allowed to access the server.  To enable access, use the Azure Management Portal or run sp_set_firewall_rule on the master database to create a firewall rule for this IP address or address range.  It may take up to five minutes for this change to take effect.

In [3]:

server_name='escwa-dev-02.escwa.un.org'
database_name='ispar'
username='ispar'
pwd='ispar_data'

connection_string = (
    f"Server={server_name};"
    f"Database={database_name};"
    f"UID={username};"
    f"PWD={pwd};"
    "TrustServerCertificate=Yes"
)

# Establish connection
with mssql_python.connect(connection_string) as conn:
    # Create a cursor object
    with conn.cursor() as cursor:
        # Execute statement to look up user tables
        cursor.execute(
            """
            SELECT TABLE_SCHEMA, TABLE_NAME 
            FROM INFORMATION_SCHEMA.TABLES 
            WHERE TABLE_TYPE = 'BASE TABLE';
            """
        )
        # Fetch and print table names
        for row in cursor.fetchall():
            print(f"Schema: {row[0]}, Table: {row[1]}")
            

Schema: dbo, Table: GenderCountryPopulation
Schema: dbo, Table: HistoricalKPI
Schema: dbo, Table: HistoricalKPI_240517
Schema: dbo, Table: HistoricalKPI_backup
Schema: dbo, Table: HistoricalKPIChange
Schema: dbo, Table: HistoricalKPIChange_240517
Schema: dbo, Table: HistoricalPeriods
Schema: dbo, Table: LNOBRegionRank
Schema: dbo, Table: Indexes_Not_On_Ispar
Schema: dbo, Table: LNOBGlobalRank
Schema: dbo, Table: Indexes_Not_On_Ispar_Data
Schema: dbo, Table: IndexImprovement
Schema: dbo, Table: IndexMax
Schema: dbo, Table: sysdiagrams
Schema: dbo, Table: KPI_Extended_Metadata
Schema: dbo, Table: temp_caps
Schema: dbo, Table: KPIHistoryAvailable
Schema: dbo, Table: KPIPolicies
Schema: dbo, Table: KPIs_b
Schema: dbo, Table: KPIs_Custom_Min_Max
Schema: dbo, Table: KPIs_test
Schema: dbo, Table: KPIsStructureType
Schema: dbo, Table: LinkBetweenPillarsPol
Schema: dbo, Table: Logs
Schema: dbo, Table: MappedKPIs
Schema: dbo, Table: KPIData_Rank_Test
Schema: dbo, Table: MappedKPIsByGender
Schema

In [7]:
import mssql_python

server_name = 'escwa-dev-02.escwa.un.org'
database_name = 'ispar'
username = 'ispar'
pwd = 'ispar_data'  # pull from env, e.g. os.environ['ISPAR_PWD']

connection_string = (
    f"Server={server_name};"
    f"Database={database_name};"
    f"UID={username};"
    f"PWD={pwd};"
    "TrustServerCertificate=Yes"
)


In [13]:
# Prod DB Pull
server_name='sqldb-p-server-001.database.windows.net'
database_name='ispar'
username='ispar_user_read'
pwd='TAescwa2025##'

connection_string = (
    f"Server={server_name};"
    f"Database={database_name};"
    f"UID={username};"
    f"PWD={pwd};"
    "TrustServerCertificate=Yes"
)

In [14]:

# Historical Data Query
placeholders = ",".join("?" for _ in kpi_list)
hist_data_query = f"""
    SELECT h.*,
           c.Name  AS CountryName,
           c.ISO   AS CountryISO,
           k.KeyID AS KPIKeyID,
           k.Name  AS KPIName
    FROM HistoricalKPI AS h
    LEFT JOIN Countries AS c
        ON h.CountryID = c.keyID
    LEFT JOIN KPIs AS k
        ON h.KPIID = k.KeyID
    WHERE h.KPIID IN ({placeholders});
"""
with mssql_python.connect(connection_string) as conn:
    with conn.cursor() as cursor:
        cursor.execute(hist_data_query, kpi_list)
        columns = [col[0] for col in cursor.description]
        rows = cursor.fetchall()
hist_data = pd.DataFrame.from_records(rows, columns=columns)

rename_map = {
        'KPIID' : 'KPI ID',
        'KPIName' : 'Series Name',
        'Year' : 'year_historical_automated',
        'Value' : 'data_historical_automated',
        'CountryISO' : 'ISO',
        'CountryName' : 'Country'
    }
hist_data = hist_data.rename(columns=rename_map)


OperationalError: Driver Error: Client unable to establish connection; DDBC Error: [Microsoft]Named Pipes Provider: Could not open a connection to SQL Server [64]. 

In [ ]:
hist_data

In [7]:
# Historical Data Query
placeholders = ",".join("?" for _ in kpi_list)
hist_data_query = f"""
    SELECT h.*,
           c.Name  AS CountryName,
           c.ISO   AS CountryISO,
           k.KeyID AS KPIKeyID,
           k.Name  AS KPIName
    FROM HistoricalKPI AS h
    LEFT JOIN Countries AS c
        ON h.CountryID = c.keyID
    LEFT JOIN KPIs AS k
        ON h.KPIID = k.KeyID
    WHERE h.KPIID IN ({placeholders});
"""
with mssql_python.connect(connection_string) as conn:
    with conn.cursor() as cursor:
        cursor.execute(hist_data_query, kpi_list)
        columns = [col[0] for col in cursor.description]
        rows = cursor.fetchall()
hist_data = pd.DataFrame.from_records(rows, columns=columns)

rename_map = {
        'KPIID' : 'KPI ID',
        'KPIName' : 'Series Name',
        'Year' : 'year_historical_automated',
        'Value' : 'data_historical_automated',
        'CountryISO' : 'ISO',
        'CountryName' : 'Country'
    }
hist_data = hist_data.rename(columns=rename_map)


OperationalError: Driver Error: Client unable to establish connection; DDBC Error: [Microsoft]Named Pipes Provider: Could not open a connection to SQL Server [53]. 

In [ ]:
# Sim Data Pull

In [2]:
# Prod DB Pull
server_name='sqldb-p-server-001.database.windows.net'
database_name='ispar'
username='ispar_user_read'
pwd='TAescwa2025##'

connection_string = (
    f"Server={server_name};"
    f"Database={database_name};"
    f"UID={username};"
    f"PWD={pwd};"
    "TrustServerCertificate=Yes"
)

In [3]:
# Dev DB Pull
server_name='escwa-dev-02.escwa.un.org'
database_name='ispar'
username='ispar'
pwd='ispar_data'

connection_string = (
    f"Server={server_name};"
    f"Database={database_name};"
    f"UID={username};"
    f"PWD={pwd};"
    "TrustServerCertificate=Yes"
)

In [53]:
index_id = '9'

kpi_structure_query = f"""
        SELECT TOP 10 * FROM KPIsStructure
        WHERE SimID = ?;
"""

with mssql_python.connect(connection_string) as conn:
    with conn.cursor() as cursor:
        cursor.execute(kpi_structure_query, (index_id,))
        columns = [col[0] for col in cursor.description]
        rows = cursor.fetchall()

country_ref_tbl = pd.DataFrame.from_records(rows, columns=columns)
#country_ref_tbl = country_ref_tbl.rename(columns={'KeyID': 'CountryID'})

In [54]:
country_ref_tbl.columns

Index(['KeyID', 'Name', 'AName', 'Description', 'ADesc', 'TypeID', 'ParentID',
       'SimID', 'Order', 'ReverseEdges'],
      dtype='str')

In [3]:
index_id = '9'

kpi_structure_query = f"""
        SELECT
        s.KeyID,
        s.Name,
        s.Description,
        s.TypeID,
        s.SimID,
        s.[Order],
        k.KeyID     AS KPI_KeyID,
        k.Name      AS KPI_Name,
        k.Description AS KPI_Description,
        k.StructureID,
        k.KPIOrder
    FROM KPIsStructure s
    LEFT JOIN KPIs k
        ON s.KeyID = k.StructureID
    WHERE s.SimID = ?
    ORDER BY s.[Order];
"""

kpi_structure_query = """
    SELECT
        s.KeyID,
        s.Name,
        s.Description,
        s.TypeID,
        s.SimID,
        s.[Order],
        k.*
    FROM KPIsStructure s
    LEFT JOIN KPIs k
        ON s.KeyID = k.StructureID
    WHERE s.SimID = ?
    ORDER BY s.[Order];
"""
kpi_structure_query = """
    SELECT
        s.KeyID        AS KeyID_Structure,
        s.Name         AS Name_Structure,
        s.AName        AS AName_Structure,
        s.Description  AS Description_Structure,
        s.ADesc        AS ADesc_Structure,
        s.TypeID       AS TypeID_Structure,
        s.ParentID     AS ParentID_Structure,
        s.SimID        AS SimID_Structure,
        s.[Order]      AS Order_Structure,
        s.ReverseEdges AS ReverseEdges_Structure,
        k.KeyID        AS KeyID_KPI,
        k.Name         AS Name_KPI,
        k.FName        AS FName_KPI,
        k.AName        AS AName_KPI,
        k.ThemeID      AS ThemeID_KPI,
        k.Description  AS Description_KPI,
        k.FDesc        AS FDesc_KPI,
        k.ADesc        AS ADesc_KPI,
        k.StructureID  AS StructureID_KPI,
        k.weight       AS weight_KPI,
        k.KPIOrder     AS KPIOrder_KPI,
        k.Editable     AS Editable_KPI,
        k.ReverseEdges AS ReverseEdges_KPI,
        k.ChangeStep   AS ChangeStep_KPI,
        k.ConsiderChangeStepInOptimization AS ConsiderChangeStepInOptimization_KPI,
        k.Min          AS Min_KPI,
        k.Max          AS Max_KPI,
        k.TAG          AS TAG_KPI,
        k.CustomMin    AS CustomMin_KPI,
        k.CustomMax    AS CustomMax_KPI,
        k.ReverseByScoreForRankCalc AS ReverseByScoreForRankCalc_KPI
    FROM KPIsStructure s
    LEFT JOIN KPIs k
        ON s.KeyID = k.StructureID
    WHERE s.SimID = ?
    ORDER BY s.[Order];
"""
with mssql_python.connect(connection_string) as conn:
    with conn.cursor() as cursor:
        cursor.execute(kpi_structure_query, (index_id,))
        columns = [col[0] for col in cursor.description]
        rows = cursor.fetchall()

country_ref_tbl = pd.DataFrame.from_records(rows, columns=columns)
#country_ref_tbl = country_ref_tbl.rename(columns={'KeyID': 'CountryID'})

In [4]:
country_ref_tbl.to_clipboard()

In [46]:
country_ref_tbl.to_clipboard()

In [30]:
index_id = '9'
# Initially query the index from the KPIsStructure DB, th
kpi_query = f"""
    SELECT Top 20 *
    FROM KPIs;"""

with mssql_python.connect(connection_string) as conn:
    with conn.cursor() as cursor:
        cursor.execute(kpi_query)
        columns = [col[0] for col in cursor.description]
        rows = cursor.fetchall()
country_ref_tbl = pd.DataFrame.from_records(rows, columns=columns)
#country_ref_tbl = country_ref_tbl.rename(columns={'keyID':'CountryID'})

In [34]:
country_ref_tbl.columns

Index(['KeyID', 'Name', 'FName', 'AName', 'ThemeID', 'Description', 'FDesc',
       'ADesc', 'StructureID', 'weight', 'KPIOrder', 'Editable',
       'ReverseEdges', 'ChangeStep', 'ConsiderChangeStepInOptimization', 'Min',
       'Max', 'TAG', 'CustomMin', 'CustomMax', 'ReverseByScoreForRankCalc'],
      dtype='str')

In [46]:
# Use StructureID to Identify KPIs
# SimID ==> 9 ==> get StructureID then query KPIID to get structure ID


In [47]:
country_ref_tbl

,SimID
0,1042
1,1048
2,9
3,1028
4,3
5,1034
6,1022
7,1051
8,12
9,1031


In [44]:
country_ref_tbl.to_clipboard()

In [33]:
country_ref_tbl.to_clipboard()

In [36]:
hist_data = hist_data.merge(country_ref_tbl).add_suffix("_db")
hist_data['Value_db'] = pd.to_numeric(hist_data['Value_db'], errors='coerce').astype('Float64')
hist_data['KPIID_db'] = hist_data['KPIID_db'].astype(str).str.strip()
hist_data['Year_db'] = hist_data['Year_db'].astype('Int64')


In [37]:
historical_kpi_data_long

,KPI ID,Series Name,year_historical_automated,ISO,data_historical_automated
0,282,Mean Year of Schooling,2022,AFG,3.0
1,282,Mean Year of Schooling,1990,AFG,1.5
2,282,Mean Year of Schooling,1991,AFG,1.6
3,282,Mean Year of Schooling,1992,AFG,1.6
4,282,Mean Year of Schooling,1993,AFG,1.7
...,...,...,...,...,...
56827,274,Active mobile-broadband subscriptions per 100 ...,2020,ZWE,56.0
56828,274,Active mobile-broadband subscriptions per 100 ...,2021,ZWE,59.1
56829,274,Active mobile-broadband subscriptions per 100 ...,2022,ZWE,60.5
56830,274,Active mobile-broadband subscriptions per 100 ...,2023,ZWE,67.6


In [47]:
print(historical_kpi_data_long.shape)
joined = historical_kpi_data_long.merge(hist_data, how='left',
                                        left_on=['KPI ID', 'year_historical_automated','ISO'],
                                        right_on=['KPIID_db','Year_db','ISO_db'])

print(joined.shape)
joined = joined[joined['Value_db'].notna()]
print(joined.shape)
joined['diff'] = abs(joined['Value_db'] - joined['data_historical_automated'])


(56832, 5)
(56832, 12)
(39325, 12)


In [49]:
joined['diff']

0        0.0
1        0.0
2        0.0
3        0.0
4        0.0
        ... 
56827    0.0
56828    0.0
56829    0.0
56830    0.0
56831    0.0
Name: diff, Length: 39325, dtype: Float64

In [60]:
t = joined[joined['diff'] > 0.00001]

In [61]:
t.to_clipboard()

In [32]:
historical_kpi_data_long['data_historical_automated'] = pd.to_numeric(
    historical_kpi_data_long['data_historical_automated'], errors='coerce').astype('Float64')
historical_kpi_data_long['KPI ID'] = historical_kpi_data_long['KPI ID'].astype(str).str.strip()
historical_kpi_data_long['year_historical_automated'] = historical_kpi_data_long['year_historical_automated'].astype('Int64')


,KPIID_db,CountryID_db,Year_db,Category_db,Value_db,Name_db,ISO_db
0,271,1,1990,1,0.00,Afghanistan,AFG
1,271,1,1991,1,0.00,Afghanistan,AFG
2,271,1,1992,1,0.00,Afghanistan,AFG
3,271,1,1993,1,0.00,Afghanistan,AFG
4,271,1,1994,1,0.00,Afghanistan,AFG
...,...,...,...,...,...,...,...
39490,1942,199,2020,1,3.35,North Macedonia,MKD
39491,1942,199,2021,1,3.52,North Macedonia,MKD
39492,1942,199,2022,1,3.23,North Macedonia,MKD
39493,1942,199,2023,1,2.92,North Macedonia,MKD


In [14]:
import mssql_python

server_name = 'escwa-dev-02.escwa.un.org'
database_name = 'ispar'
username = 'ispar'
pwd = 'ispar_data'  # pull from env, e.g. os.environ['ISPAR_PWD']

connection_string = (
    f"Server={server_name};"
    f"Database={database_name};"
    f"UID={username};"
    f"PWD={pwd};"
    "TrustServerCertificate=Yes"
)



In [49]:
kpi_info.to_clipboard()

In [16]:
countries

,keyID,Name,ISO
0,1,Afghanistan,AFG
1,2,Albania,ALB
2,3,Algeria,DZA
3,4,Andorra,AND
4,5,Angola,AGO
...,...,...,...
198,207,Kosovo,KSV
199,208,Puerto Rico,PRI
200,209,Taiwan,TWN
201,211,Montserrat,MSR
